# Dashboard Immobilier — Tableau de bord de l'attractivité

Ce notebook répond à l'exigence du cahier des charges imposant une présentation sous
forme de notebook Jupyter. Il reproduit le dashboard de `app/pages/1_Dashboard.py`
(vue d'ensemble, Top 10 des communes, cartes Folium, choroplèthes par département),
avec des filtres interactifs (`ipywidgets`) équivalents au panneau latéral Streamlit.

Il lit uniquement les CSV déjà calculés dans `data/final/` — il ne relance aucune
extraction ni transformation. Si `data/final/` est vide ou périmé, lance d'abord
[`etl_pipeline.ipynb`](etl_pipeline.ipynb) (au moins une fois).

**Question métier du projet** : dans quelle ville française acheter pour investir ?

`score_attractivite` = 35% rendement brut + 25% ratio d'effort fiscal (inversé)
+ 20% revenu fiscal moyen + 20% taux de vacance (inversé), chaque composante
normalisée de 0 à 100 au niveau national.

## 0. Chargement des données

Si le kernel `immobilier` n'apparaît pas dans Jupyter, enregistre-le une fois
(environnement conda actif) :

```bash
python -m ipykernel install --user --name immobilier --display-name "Python (immobilier)"
```

In [1]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import folium
from IPython.display import display

# Localise la racine du projet (ce notebook peut être lancé depuis notebooks/ ou
# depuis la racine selon la façon dont Jupyter est démarré).
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        PROJECT_ROOT = candidate
        break

pd.set_option("display.max_columns", 30)

FINAL_DIR = PROJECT_ROOT / "data" / "final"
score = pd.read_csv(FINAL_DIR / "score_attractivite.csv")
communes = pd.read_csv(FINAL_DIR / "communes.csv")
demographics = pd.read_csv(FINAL_DIR / "demographics.csv")
transactions = pd.read_csv(FINAL_DIR / "transactions.csv", parse_dates=["date_transaction"])

print(f"Lu depuis {FINAL_DIR} : score={len(score):,} | communes={len(communes):,} | transactions={len(transactions):,}")

Lu depuis /Users/damienmolinieres/Documents/GitHub/etl_immobilier/data/final : score=20,198 | communes=34,969 | transactions=3,110,694


## Construction du dashboard

Reconstruit les mêmes indicateurs et cartes que `app/pages/1_Dashboard.py` directement
à partir des CSV lus ci-dessus (pas besoin que PostgreSQL soit démarré).

In [2]:
df = (
    score.dropna(subset=["score_attractivite"])
    .merge(communes, on="id_ville", how="inner")
    .sort_values("score_attractivite", ascending=False)
    .reset_index(drop=True)
)
df = df[[
    "ville", "departement", "code_insee", "latitude", "longitude",
    "prix_m2_median", "loyer_m2_moyen", "rendement_brut", "taux_vacance",
    "ratio_effort_fiscal", "score_attractivite", "n_transactions",
]]
df["departement"] = df["departement"].astype(str).str.strip()

densite_dept = (
    communes.merge(demographics, on="id_ville")
    .assign(departement=lambda d: d["departement"].astype(str).str.strip())
    .groupby("departement")["densite"].mean()
    .reset_index()
)

annees_disponibles = sorted(transactions["annee"].unique().tolist(), reverse=True)
annee_ref = int(score["annee_ref"].iloc[0])
derniere_annee_prix = max(annees_disponibles)

prix_dept_derniere_annee = (
    transactions[transactions["annee"] == derniere_annee_prix]
    .merge(communes[["id_ville", "departement"]], on="id_ville")
    .assign(departement=lambda d: d["departement"].astype(str).str.strip())
    .groupby("departement")["prix_m2"].median()
    .reset_index()
    .rename(columns={"prix_m2": "prix_m2_median"})
)

print(
    f"{len(df):,} communes évaluées \u2014 prix/rendement {derniere_annee_prix}, "
    f"loyer/revenu fiscal/vacance {annee_ref}"
)

13,109 communes évaluées — prix/rendement 2025, loyer/revenu fiscal/vacance 2024


In [3]:
def render_commune_map(data: pd.DataFrame, value_col: str, colors: list[str], legend: str, map_size: int) -> folium.Map:
    map_data = data.dropna(subset=["latitude", "longitude", value_col]).head(map_size)
    colormap = folium.LinearColormap(
        colors=colors, vmin=data[value_col].min(), vmax=data[value_col].max(), caption=legend
    )
    france_map = folium.Map(location=[46.6, 2.6], zoom_start=6, tiles="cartodbpositron")
    for _, row in map_data.iterrows():
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=5,
            color=colormap(row[value_col]),
            fill=True,
            fill_color=colormap(row[value_col]),
            fill_opacity=0.85,
            popup=(
                f"<b>{row['ville']}</b> ({row['departement']})<br>"
                f"{legend} : {row[value_col]:.2f}<br>"
                f"Score : {row['score_attractivite']:.1f}/100"
            ),
        ).add_to(france_map)
    colormap.add_to(france_map)
    return france_map


DEPARTEMENTS_GEOJSON = "https://france-geojson.gregoiredavid.fr/repo/departements.geojson"


def render_choropleth(data: pd.DataFrame, columns: list[str], fill_color: str, legend: str) -> folium.Map:
    carte = folium.Map(location=[46.6, 2.6], zoom_start=5, tiles="cartodbpositron")
    choropleth = folium.Choropleth(
        geo_data=DEPARTEMENTS_GEOJSON,
        data=data,
        columns=columns,
        key_on="feature.properties.code",
        fill_color=fill_color,
        fill_opacity=0.8,
        line_opacity=0.3,
        nan_fill_color="white",
        legend_name=legend,
    ).add_to(carte)
    choropleth.geojson.add_child(folium.GeoJsonTooltip(fields=["nom"], aliases=["Département :"]))
    return carte

### Filtres interactifs

Équivalent notebook du panneau latéral Streamlit : choisis un ou plusieurs
départements, la taille des cartes et le rang (dans le Top 10 courant) affiché par le
pie chart occupé/vacant, puis clique sur **Run Interact** pour rafraîchir tout le
dashboard ci-dessous.

In [4]:
from ipywidgets import interact_manual, widgets


@interact_manual(
    departements=widgets.SelectMultiple(
        options=sorted(df["departement"].unique()),
        value=(),
        description="Départements",
        layout=widgets.Layout(width="320px"),
    ),
    map_size=widgets.IntSlider(
        min=10, max=min(2000, len(df)), step=10, value=min(200, len(df)),
        description="Communes/carte",
    ),
    rang_commune=widgets.IntSlider(min=1, max=10, step=1, value=1, description="Rang (pie)"),
)
def update_dashboard(departements, map_size, rang_commune):
    df_filtered = df[df["departement"].isin(departements)] if departements else df
    top_10 = df_filtered.head(10)

    print("Vue d'ensemble")
    print(f"  Communes évaluées                              : {len(df_filtered):,}")
    print(f"  Score moyen                                     : {df_filtered['score_attractivite'].mean():.1f}/100")
    print(f"  Rendement brut moyen ({derniere_annee_prix}/{annee_ref})                 : {df_filtered['rendement_brut'].mean():.2f} %")
    print(f"  Taux de vacance moyen ({annee_ref})                        : {df_filtered['taux_vacance'].mean():.2f} %")

    print(f"\nTop 10 des communes les plus attractives (\u2265 5 transactions en {derniere_annee_prix})")
    top_10_display = top_10[[
        "ville", "departement", "prix_m2_median", "loyer_m2_moyen", "rendement_brut",
        "taux_vacance", "ratio_effort_fiscal", "n_transactions", "score_attractivite",
    ]]
    display(
        top_10_display.style.format({
            "prix_m2_median": "{:.0f} \u20ac",
            "loyer_m2_moyen": "{:.2f} \u20ac",
            "rendement_brut": "{:.2f} %",
            "taux_vacance": "{:.2f} %",
            "ratio_effort_fiscal": "{:.2f}",
            "score_attractivite": "{:.1f}",
        }).bar(subset=["score_attractivite"], color="#1a9850", vmin=0, vmax=100)
    )

    print("\nCarte du score d'attractivité")
    display(render_commune_map(
        df_filtered, "score_attractivite", ["#d73027", "#fee08b", "#1a9850"],
        "Score d'attractivité (0-100)", map_size,
    ))

    print("\nCarte du ratio d'effort fiscal (plus bas = mieux)")
    display(render_commune_map(
        df_filtered, "ratio_effort_fiscal", ["#1a9850", "#fee08b", "#d73027"],
        "Ratio d'effort fiscal", map_size,
    ))

    if not top_10.empty:
        rang = min(rang_commune, len(top_10)) - 1
        commune_choisie = top_10.iloc[rang]
        print(f"\nOccup\u00e9 vs vacant \u2014 {commune_choisie['ville']} (donn\u00e9es {annee_ref})")
        fig, ax = plt.subplots(figsize=(4, 4))
        taux = commune_choisie["taux_vacance"]
        ax.pie(
            [100 - taux, taux], labels=["Occup\u00e9", "Vacant"], autopct="%.1f%%",
            colors=["#1a9850", "#d73027"], startangle=90,
        )
        ax.axis("equal")
        plt.show()

    prix_dept_filtre = (
        prix_dept_derniere_annee[prix_dept_derniere_annee["departement"].isin(departements)]
        if departements else prix_dept_derniere_annee
    )
    densite_dept_filtre = (
        densite_dept[densite_dept["departement"].isin(departements)]
        if departements else densite_dept
    )

    print(f"\nPrix m\u00e9dian au m\u00b2 par d\u00e9partement \u2014 {derniere_annee_prix}")
    display(render_choropleth(prix_dept_filtre, ["departement", "prix_m2_median"], "YlOrRd", "Prix médian (€/m²)"))

    print("\nDensit\u00e9 de population par d\u00e9partement (hab/km\u00b2)")
    display(render_choropleth(densite_dept_filtre, ["departement", "densite"], "PuBu", "Densité (habitants/km²)"))

interactive(children=(SelectMultiple(description='Départements', layout=Layout(width='320px'), options=('01', …

---
*Pour l'application interactive complète (Streamlit), voir `app/app.py`. Pour
régénérer `data/final/*.csv`, voir [`etl_pipeline.ipynb`](etl_pipeline.ipynb).*